# Lab 04 — Build the Feed

## The story

Open Instagram, TikTok or X: the first thing you see is a **feed** — a list of posts, in a specific order. Somebody (well, some *algorithm*) decided which posts you see and which one comes first. In InstaKill that list does not exist yet. This week you build the brain behind it.

There is **no AI model** in this lab. A feed can already be good with counting, simple math, and one clever algorithm.

## How this notebook works

Almost everything here is **already implemented**. Your job is to *read it, run it, and understand why it is built this way* — each step explains what it does and where it will appear in InstaKill.

At the end you get **one task**: tune three "knobs" (the weights) so that your feed beats the default settings and the two simple baselines. You do not need to write any function.

## The path

1. Look at the **past** (what users liked, how new an item is, what is popular) and turn it into three **signals**.
2. Mix the signals into one **score** per item and user.
3. Pick the **top 10** items efficiently.
4. See how the feed behaves with 100× more items, and why.
5. **Your task:** tune the weights so the feed gets better.

## Why movies, and not InstaKill posts?

To know whether a feed is good, we need thousands of real likes. InstaKill only has a handful of seeded posts. So we practise on MovieLens (real movie ratings) and then you move the same ideas into your `GET /feed` endpoint.

| In this notebook | In InstaKill |
|---|---|
| movie | post |
| user | user |
| rating of 4 or more | a like |
| the first time anybody rated the movie | the moment the post was created |

**Rules:** CPU only; Python standard library + NumPy; no `sklearn`, `surprise`, `implicit`, embeddings, or other recommender library.

In [ ]:
%pip -q install numpy

In [ ]:
import csv
import heapq
import random
import time
import tracemalloc
import zipfile
from collections import Counter, defaultdict, namedtuple
from dataclasses import dataclass
from pathlib import Path

import numpy as np

SEED = 42
K = 10                      # how many items a feed shows
DAY = 86_400                # seconds in a day
HALF_LIFE_DAYS = 3 * 365    # an item loses half of its freshness every 3 years (you may change this later)
POPULARITY_DAYS = 365       # popularity counts only the likes of the last year (you may change this later)
DATA_DIR = Path("ml-latest-small")

## Data

MovieLens `latest-small` is a public dataset of about 100 000 movie ratings from 600 users. It comes from GroupLens and is **not** stored in the course repository. Read its terms of use in `README.txt` inside the zip.

**Step 1 — download it in your browser:** <https://files.grouplens.org/datasets/movielens/ml-latest-small.zip> (about 1 MB).

Why not download it from the notebook? The GroupLens server has a certificate problem that Python rejects (your browser copes with it). So:

- **Colab:** run the next cell and pick the zip when asked.
- **Local Jupyter:** put the zip next to this notebook and run the next cell.

The cell also turns the ratings into simple records: a `Like` is *(user, item, time)* and an `Item` knows when it first appeared, its genres and its title.

In [ ]:
if not (DATA_DIR / "ratings.csv").exists():
    archives = sorted(Path(".").glob("ml-latest-small*.zip"))
    if not archives:
        try:
            from google.colab import files
            print("Select the ml-latest-small.zip file you downloaded.")
            files.upload()
        except ImportError:
            raise FileNotFoundError("Download ml-latest-small.zip (link above) and place it next to this notebook.")
        archives = sorted(Path(".").glob("ml-latest-small*.zip"))
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(".")
for name in ("ratings.csv", "movies.csv"):
    assert (DATA_DIR / name).exists(), f"{DATA_DIR / name} not found: upload the original ml-latest-small.zip"

Like = namedtuple("Like", "user_id item_id ts")                  # one positive interaction
Item = namedtuple("Item", "item_id first_seen genres title")     # first_seen = time of the first rating (seconds)

with open(DATA_DIR / "ratings.csv", newline="") as f:
    EVENTS = [(int(r["userId"]), int(r["movieId"]), int(r["timestamp"]), float(r["rating"]))
              for r in csv.DictReader(f)]
with open(DATA_DIR / "movies.csv", newline="") as f:
    MOVIES = {int(r["movieId"]): (tuple(g for g in r["genres"].split("|") if g != "(no genres listed)"), r["title"])
              for r in csv.DictReader(f)}

first_seen = {}
for _, item_id, ts, _ in EVENTS:
    first_seen[item_id] = min(ts, first_seen.get(item_id, ts))

ITEMS = {i: Item(i, first_seen[i], *MOVIES.get(i, ((), "?"))) for i in sorted(first_seen)}
LIKES = sorted((Like(u, i, ts) for u, i, ts, rating in EVENTS if rating >= 4.0),
               key=lambda like: (like.ts, like.user_id, like.item_id))
print(f"{len(LIKES):,} likes, {len(ITEMS):,} items, {len({l.user_id for l in LIKES})} users")

## How do we know a feed is good? Hide the future

Imagine today is a Monday. A real app only knows what happened **before** Monday. So we test the same way:

1. Pick a moment in time, `now`.
2. Give the feed only what happened **before** `now` — this is the `history`.
3. Look at what each user *really* liked **after** `now`. A good feed would have shown them exactly those items.

Three words you will see all the time:

| Word | Meaning |
|---|---|
| **candidates** | the items we are allowed to show a user: they already existed at `now`, and the user has not liked them yet |
| **relevant** | the candidates the user really liked after `now` (the "right answers") |
| **history** | all likes before `now` — the **only** thing your code may learn from |

We make two such tests, like a practice exam and a real exam:

- `val` — use it as much as you want while you build and tune your feed;
- `test` — the final exam. You look at it **once**, at the end.

Run the next cell and read the output; there is nothing to change.

In [ ]:
@dataclass(frozen=True)
class Split:
    name: str
    now: int
    history: list     # list[Like] with ts < now
    users: list       # evaluation users (sorted)
    relevant: dict    # user_id -> frozenset(item_id); the "right answers", never an input of your feed


def make_split(name, now, until):
    history = [like for like in LIKES if like.ts < now]
    liked = defaultdict(set)
    for like in history:
        liked[like.user_id].add(like.item_id)
    relevant = defaultdict(set)
    for like in LIKES:
        if now <= like.ts < until and ITEMS[like.item_id].first_seen < now and like.item_id not in liked[like.user_id]:
            relevant[like.user_id].add(like.item_id)
    history_size = Counter(like.user_id for like in history)
    users = sorted(u for u in relevant if history_size[u] >= 20 and len(relevant[u]) >= 3)
    if len(users) > 200:
        users = sorted(random.Random(SEED).sample(users, 200))
    return Split(name, now, history, users, {u: frozenset(relevant[u]) for u in users})


timestamps = np.array([like.ts for like in LIKES])
T_VAL, T_TEST = int(np.quantile(timestamps, 0.70)), int(np.quantile(timestamps, 0.85))
VAL = make_split("val", T_VAL, T_TEST)
TEST = make_split("test", T_TEST, int(timestamps.max()) + 1)
print(f"val:  {len(VAL.history):,} likes of history, {len(VAL.users)} users to evaluate")
print(f"test: {len(TEST.history):,} likes of history, {len(TEST.users)} users to evaluate")

## Is my feed any good? Baselines and a score

A **baseline** is a very simple feed that we try to beat. If your clever feed cannot beat a dumb one, the cleverness is not worth its cost. We give you two:

- **chronological** — newest items first (what early social apps did);
- **popularity** — the items with the most likes, the same for everybody.

To compare feeds we use two numbers (both between 0 and 1, higher is better):

- **NDCG@10** — "are the things the user will like near the **top** of the 10 shown?" 1 means a perfect feed, 0 means nothing relevant in the top 10. A good item at position 1 counts more than at position 10. This is our main number.
- **HitRate@10** — "what share of users got **at least one** good item in their 10?"

The harness also protects you from mistakes: it raises an error if a feed shows more than 10 items, repeats an item, shows something the user already liked, or shows something that did not exist yet.

Run the cell and note the two baseline scores — they are the numbers to beat.

In [ ]:
def ndcg_at_k(ranked, relevant, k=K):
    dcg = sum(1 / np.log2(rank + 2) for rank, item in enumerate(ranked[:k]) if item in relevant)
    ideal = sum(1 / np.log2(rank + 2) for rank in range(min(k, len(relevant))))
    return dcg / ideal


def evaluate(feed, split, k=K):
    """`feed` is a function: user_id -> list of item ids."""
    liked = defaultdict(set)
    for like in split.history:
        liked[like.user_id].add(like.item_id)
    scores, hits, per_user = [], [], {}
    started = time.perf_counter()
    for user in split.users:
        ranked = list(feed(user))
        if len(ranked) > k or len(set(ranked)) != len(ranked):
            raise ValueError(f"user {user}: feed must have at most {k} distinct items")
        if any(item in liked[user] for item in ranked):
            raise ValueError(f"user {user}: feed contains an item the user already liked")
        if any(ITEMS[item].first_seen >= split.now for item in ranked):
            raise ValueError(f"user {user}: feed contains an item that did not exist yet")
        per_user[user] = ndcg_at_k(ranked, split.relevant[user], k)
        scores.append(per_user[user])
        hits.append(any(item in split.relevant[user] for item in ranked))
    return {"ndcg": float(np.mean(scores)), "hit_rate": float(np.mean(hits)),
            "seconds": time.perf_counter() - started, "per_user": per_user}


def chronological_feed(split):
    """Baseline 1: newest items first."""
    newest = sorted((i for i, item in ITEMS.items() if item.first_seen < split.now),
                    key=lambda i: (-ITEMS[i].first_seen, i))
    liked = defaultdict(set)
    for like in split.history:
        liked[like.user_id].add(like.item_id)
    return lambda user: [i for i in newest if i not in liked[user]][:K]


def popularity_feed(split):
    """Baseline 2: most-liked items overall (the same list for everybody)."""
    counts = Counter(like.item_id for like in split.history)
    ranked = sorted(counts, key=lambda i: (-counts[i], i))
    liked = defaultdict(set)
    for like in split.history:
        liked[like.user_id].add(like.item_id)
    return lambda user: [i for i in ranked if i not in liked[user]][:K]


for name, make in [("chronological", chronological_feed), ("popularity", popularity_feed)]:
    result = evaluate(make(VAL), VAL)
    print(f"{name:14s} val NDCG@10={result['ndcg']:.4f}  HitRate@10={result['hit_rate']:.3f}")

## Step 1 — Signals: what can the past tell us?

A **signal** is a number that says something about an item for a user. Your feed will use three:

| Signal | Plain question | Example |
|---|---|---|
| **freshness** | how new is this item? | a post from an hour ago is fresher than one from last year |
| **affinity** | does it match what *this user* likes? | the user liked 8 comedies, so another comedy fits them well |
| **popularity** | are other people liking it lately? | 300 likes in the last year vs. 2 |

In InstaKill: freshness comes from `created_at`, popularity from the like table, and affinity from what the user liked or who they follow.

**Two rules**

1. **Only the past.** Use only likes with `ts < now`. If tomorrow's likes leak in, your score looks great here but could never work in a real app — this mistake is called **data leakage**, and the notebook checks for it.
2. **Same scale.** Every signal must land between 0 and 1. Otherwise a signal measured in thousands would drown out one measured in fractions, whatever the weights say.

**Why `build_signals`?** Think of cooking: you chop the vegetables once, not again for every plate. `build_signals` does the heavy counting **once**; later, scoring one item is a quick lookup.

`build_signals` loops over the history and fills three containers:

| Container | Holds |
|---|---|
| `liked` | for each user, the set of items they liked |
| `genre_counts` | for each user, how many of their liked items have each genre, e.g. `{"Comedy": 8, "Drama": 2}` |
| `recent_likes` | for each item, how many likes it got in the last `POPULARITY_DAYS` days |

Read the function below: the loop, the "ignore the future" rule, and the three updates are in `build_signals`.

In [ ]:
Signals = namedtuple("Signals", "liked genre_counts recent_likes max_recent")


def build_signals(history, items, now):
    liked = defaultdict(set)
    genre_counts = defaultdict(Counter)
    recent_likes = Counter()
    for like in history:
        if like.ts >= now:
            continue
        liked[like.user_id].add(like.item_id)
        for genre in items[like.item_id].genres:
            genre_counts[like.user_id][genre] += 1
        if like.ts >= now - POPULARITY_DAYS * DAY:
            recent_likes[like.item_id] += 1
    max_recent = max(recent_likes.values(), default=1)
    return Signals(liked, genre_counts, recent_likes, max_recent)

## Step 2 — The three signals, one small function each

Each function returns a number between 0 and 1.

- **freshness** — `1.0` for a brand-new item, `0.5` after `HALF_LIFE_DAYS` days, `0.25` after twice that, and so on. The formula is `0.5 ** (age_days / HALF_LIFE_DAYS)`. The age of an item is `now - item.first_seen` seconds.
- **affinity** (careful: the function receives ONE item, called `item`; `items` with an s is the dictionary of all items) — take the user's genre counts. Add up the counts of the genres that the item has (`matching`), and divide by the user's total count times the number of genres of the item. Example: the user has `{"Drama": 3, "Comedy": 1}` and the item is a Drama → `3 / (4 × 1) = 0.75`. A user we know nothing about gets `0.0`.
- **popularity** — likes of this item in the last year, divided by the likes of the most popular item (`signals.max_recent`).

In [ ]:
def freshness(item, now):
    age_days = (now - item.first_seen) / DAY
    return 0.5 ** (age_days / HALF_LIFE_DAYS)


def affinity(user_id, item, signals):
    counts = signals.genre_counts.get(user_id)
    if not counts or not item.genres:
        return 0.0
    total = sum(counts.values())
    matching = sum(counts[genre] for genre in item.genres)
    return matching / (total * len(item.genres))


def popularity(item_id, signals):
    return signals.recent_likes[item_id] / signals.max_recent

## Self-check 1 — do the signals work?

This runs your functions on a tiny hand-made example where we know the right answer.

In [ ]:
Drama = Item(1, 0, ("Drama",), "tiny drama")

# freshness: 1.0 now, 0.5 after one half-life
assert abs(freshness(Drama, now=0) - 1.0) < 1e-9
assert abs(freshness(Drama, now=HALF_LIFE_DAYS * DAY) - 0.5) < 1e-9

# affinity and popularity on a hand-made Signals object
tiny = Signals(liked={}, genre_counts={7: Counter({"Drama": 3, "Comedy": 1})}, recent_likes=Counter({1: 2}), max_recent=4)
assert abs(affinity(7, Drama, tiny) - 0.75) < 1e-9, "user 7 has 3 of 4 genre-likes in Drama"
assert affinity(8, Drama, tiny) == 0.0, "unknown user -> 0.0"
assert abs(popularity(1, tiny) - 0.5) < 1e-9 and popularity(99, tiny) == 0.0

# build_signals: the like at ts=300 is in the future (now=250) and must be ignored
items = {10: Item(10, 50, ("Drama",), "a"), 11: Item(11, 60, ("Comedy",), "b")}
history = [Like(1, 10, 100), Like(1, 11, 200), Like(2, 10, 300)]
signals = build_signals(history, items, now=250)
assert signals.liked[1] == {10, 11} and 2 not in signals.liked, "the like from the future leaked in"
assert dict(signals.genre_counts[1]) == {"Drama": 1, "Comedy": 1}
assert dict(signals.recent_likes) == {10: 1, 11: 1}
print("Self-check 1 passed.")

## Step 3 — Score one item

Now we mix the three signals into **one number**: higher means "show this earlier to this user".

```text
score = w_freshness × freshness + w_affinity × affinity + w_popularity × popularity
```

The `w_...` numbers are the **weights**: volume knobs. Raise the affinity weight and the feed becomes more personal; raise the popularity weight and everybody sees more of the same hits.

`weights` is a normal dictionary that you **pass in** as an argument, e.g. `{"freshness": 1.0, "affinity": 2.0, "popularity": 0.5}`. There is no hidden global variable, so you can try different weights side by side — we do that in the tuning step.

`score_item` should be **pure**, as in Week 2: the same inputs always give the same score, and it reads no files and changes nothing. In InstaKill this becomes the core of your `rank_feed` function, which is why it must be easy to test.

In [ ]:
def score_item(user_id, item_id, signals, items, now, weights):
    item = items[item_id]
    f = freshness(item, now)
    a = affinity(user_id, item, signals)
    p = popularity(item_id, signals)
    return weights["freshness"] * f + weights["affinity"] * a + weights["popularity"] * p

## Step 4 — Pick the top 10 without sorting everything

You will have thousands of scored items and show only 10. The obvious way — sort them all, take 10 — works, but wastes effort: you put item #9 000 in order just to throw it away.

**A cheaper idea:** look at the scores one at a time and keep only the 10 best so far. For each new score, compare it with the *worst* of your current 10; if it is better, swap them. At the end, your 10 are the winners. Python's `heapq` module does this for you:

```python
import heapq
heapq.nsmallest(2, ["pear", "fig", "banana"], key=len)   # -> ["fig", "pear"]  (the 2 shortest words)
```

`nsmallest(k, things, key=...)` returns the `k` things with the **smallest** key. We want the **biggest** scores, so the trick is to use the key `(-score, item_id)`: the minus sign turns "biggest score" into "smallest key", and when two scores are equal, the **lower `item_id`** comes first. Equal input then always gives equal output (like the tie-break in Week 3), which makes bugs reproducible.

The input is a stream of `(score, item_id)` pairs, and the result is only the list of `item_id`s.

In InstaKill: whenever a user opens the app you may have thousands of candidate posts; this function decides which 20 are shown, on every request.

In [ ]:
def top_k(scored, k):
    best = heapq.nsmallest(k, scored, key=lambda pair: (-pair[0], pair[1]))
    return [item_id for score, item_id in best]

## Self-check 2 — does `top_k` work?

The last check feeds 200 000 pairs through `top_k` and measures memory — if you store or sort everything instead of keeping only the best `k`, it fails.

In [ ]:
assert top_k([], 3) == [] and top_k([(1.0, 1)], 0) == []
assert top_k([(0.5, 9), (0.9, 4), (0.9, 2), (0.1, 1)], 3) == [2, 4, 9], "ties: lower item_id first"
assert top_k([(0.5, 1), (0.4, 2)], 10) == [1, 2], "k larger than the input"

tracemalloc.start()
stream = ((((i * 7919) % 1000) / 1000, i) for i in range(200_000))   # a generator: can be read only once
best = top_k(stream, 10)
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
assert len(best) == 10
assert peak < 1_000_000, f"top_k kept {peak/1e6:.1f} MB for k=10: do not store or sort the whole input"
print("Self-check 2 passed.")

## Step 5 — The whole pipeline: candidates → score → top-k

Every feed in the world does the same three steps:

```text
 all items ──▶ [1 CANDIDATES] ──▶ [2 SCORE] ──▶ [3 TOP-K] ──▶ the feed
               "what may we       "how good     "which 10 do
                show this user?"   is each one?"  we show?"
```

1. **Candidates** — items that existed at `now` (`first_seen < now`) and that the user has not liked yet.
2. **Score** — call `score_item` for each candidate, producing `(score, item_id)` pairs.
3. **Top-k** — pass the pairs to `top_k`.

This is exactly what your InstaKill `GET /feed` will do: fetch candidate posts from the database, score them, return the best `limit`.

In [ ]:
def build_feed(user_id, signals, items, now, weights, k=K):
    already_liked = signals.liked.get(user_id, set())
    candidates = [i for i, item in items.items() if item.first_seen < now and i not in already_liked]
    scored = [(score_item(user_id, i, signals, items, now, weights), i) for i in candidates]
    return top_k(scored, k)

## Look at a real feed

Pick a user, see what they liked, and what the feed (with equal weights for the three signals) shows them. Does it look sensible? Try a few users: change the number in `VAL.users[...]`.

In [ ]:
signals_val = build_signals(VAL.history, ITEMS, VAL.now)
default_weights = {"freshness": 1.0, "affinity": 1.0, "popularity": 1.0}


def show_feed(user, weights=default_weights):
    liked_before = [ITEMS[like.item_id] for like in VAL.history if like.user_id == user][-5:]
    print(f"User {user} liked recently:")
    for item in liked_before:
        print("   ", item.title, item.genres)
    print("Feed:")
    for item_id in build_feed(user, signals_val, ITEMS, VAL.now, weights):
        print("   ", ITEMS[item_id].title, ITEMS[item_id].genres)


show_feed(VAL.users[0])

## Self-check 3 — is the feed fair and repeatable?

It checks three things:

- **Deterministic:** asking twice gives the same feed.
- **Sensible:** every feed has between 1 and 10 items, and every score is a real number.
- **No cheating (leakage):** the test secretly adds *future* likes to the history. If your feed changes, your code used information from after `now` — that is the mistake `build_signals` avoids with its `ts >= now` rule.

In [ ]:
sample_users = VAL.users[:20]
first = {u: build_feed(u, signals_val, ITEMS, VAL.now, default_weights) for u in sample_users}
assert first == {u: build_feed(u, signals_val, ITEMS, VAL.now, default_weights) for u in sample_users}, "feed must be deterministic"
assert all(0 < len(feed) <= K for feed in first.values())

future_likes = [like for like in LIKES if like.ts >= VAL.now]
leaky_signals = build_signals(VAL.history + future_likes, ITEMS, VAL.now)
assert first == {u: build_feed(u, leaky_signals, ITEMS, VAL.now, default_weights) for u in sample_users}, \
    "leakage: your feed changed when events from the future were added to the history"

assert all(np.isfinite(score_item(u, i, signals_val, ITEMS, VAL.now, default_weights))
           for u in sample_users for i in list(ITEMS)[:200])
print("Self-check 3 passed.")

In [ ]:
def make_feed(split, signals, weights):
    """Turns build_feed into the `user -> list of items` function that evaluate() expects."""
    return lambda user: build_feed(user, signals, ITEMS, split.now, weights)


def val_ndcg(weights):
    return evaluate(make_feed(VAL, signals_val, weights), VAL)["ndcg"]


baseline_val = {name: evaluate(make(VAL), VAL)["ndcg"]
                for name, make in [("chronological", chronological_feed), ("popularity", popularity_feed)]}
default_val = val_ndcg(default_weights)

## Which signals really matter? (ablation)

"Ablation" means *switch one thing off and see what happens*. The cell below scores the feed on `val` with all three signals, and then with each signal set to `0`.

Read it like this:

- the score **drops a lot** when a signal is off → that signal is **important** (it earns its place);
- the score **barely changes** → that signal does little; you could drop it from InstaKill and save code and time;
- the score **goes up** when a signal is off → that signal is **hurting**.

**Keep this table in mind: it is the main clue for your task at the end.**

In [ ]:
variants = {
    "all three signals": default_weights,
    "without freshness": {**default_weights, "freshness": 0.0},
    "without affinity": {**default_weights, "affinity": 0.0},
    "without popularity": {**default_weights, "popularity": 0.0},
}
ablation = {name: val_ndcg(w) for name, w in variants.items()}
for name, value in ablation.items():
    print(f"{name:20s} val NDCG@10 = {value:.4f}   {'█' * round(60 * value)}")

## Does it still work with 100× more items? (Complexity)

Your feed is fast on 7 000 movies. InstaKill may one day have a million posts. **Complexity** (written *Big-O*) answers one question: *when the data grows, how much slower does the code get?*

| Data grows ×10, time grows… | Name | Feeling |
|---|---|---|
| ×10 | linear, `O(n)` | fine |
| ×100 | quadratic, `O(n²)` | the app dies as it grows |

Rules of thumb: one loop over `n` items is `O(n)`; sorting `n` items is `O(n log n)`; keeping a best-`k` list while looping is `O(n log k)`.

The cell below times one `build_feed` call with 1 000, 10 000 and 100 000 items (`scaled_catalog(n)` creates `n` made-up items), then compares `top_k` with "sort everything, take 10". Run it and read the numbers.

In [ ]:
def scaled_catalog(n):
    rng = random.Random(SEED)
    genre_pool = sorted({g for item in ITEMS.values() for g in item.genres})
    catalog = {}
    for i in range(1, n + 1):
        item_id = 10_000_000 + i
        catalog[item_id] = Item(item_id, VAL.now - rng.randrange(1, 3000) * DAY,
                                tuple(rng.sample(genre_pool, rng.randint(1, 3))), "made-up item")
    return catalog


def mean_seconds(fn, repeats=5):
    """Runs fn() `repeats` times and returns the average time in seconds."""
    started = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - started) / repeats


user = VAL.users[0]
timings = {}
for n in (1_000, 10_000, 100_000):
    catalog = scaled_catalog(n)
    catalog_signals = build_signals(VAL.history, {**ITEMS, **catalog}, VAL.now)
    timings[n] = mean_seconds(lambda: build_feed(user, catalog_signals, catalog, VAL.now, default_weights))
    print(f"{n:>7,} items: build_feed takes {timings[n]*1000:7.1f} ms")
print(f"-> 100x more items took {timings[100_000] / timings[1_000]:.0f}x longer (linear growth would be about 100x).")

pairs = [(score_item(user, i, catalog_signals, catalog, VAL.now, default_weights), i) for i in catalog]
topk_ms = 1000 * mean_seconds(lambda: top_k(pairs, K))
sorted_ms = 1000 * mean_seconds(lambda: sorted(pairs, key=lambda p: (-p[0], p[1]))[:K])
print(f"top 10 of 100 000 scores: top_k {topk_ms:.1f} ms  vs  sort everything {sorted_ms:.1f} ms")

**What the numbers say**

- `build_signals` reads each like once: it is `O(H)`, where `H` is the number of likes in the history. It runs **once**, not per request.
- `build_feed` looks at every candidate once (`C` of them) and the heap keeps only the best `k`: it is `O(C log k)`. Going from 1 000 to 100 000 items made it about 100× slower — linear, as expected.
- Sorting everything would be `O(C log C)`. The gap between `top_k` and sorting is small here and grows with the number of items.
- The most expensive part of a request is **scoring every candidate**. If InstaKill gets slow, the first thing to try is computing freshness and popularity once per item (not once per user) and scoring only a smaller set of candidates — the idea behind the optimisation week later in the course.

## Your task — tune the three knobs

The feed mixes three signals with three weights:

```text
score = w_freshness × freshness + w_affinity × affinity + w_popularity × popularity
```

Think of them as volume knobs:

| Knob | Turn it up and… |
|---|---|
| `freshness` | newer items climb to the top |
| `affinity` | the feed matches each user's own taste more |
| `popularity` | everybody sees more of the same hits |

`0` switches a signal off, `1` is normal, `3` means "three times as important". Only the *proportions* matter: `(2, 2, 2)` behaves exactly like `(1, 1, 1)`.

**Goal:** find weights that score **higher on `val` than the default `(1, 1, 1)`** and than both baselines.

**How to approach it**

1. Run the cell below as it is: you see the starting point (the default weights).
2. Look at the **ablation table** above. The signal whose removal hurt the most is important: turn its knob **up**. A signal whose removal made the score **go up** hurts: turn its knob **down**, even to `0`.
3. Change **one number at a time**, run the cell, and keep the change only if your bar gets longer.
4. A handful of tries is enough. Do not chase tiny differences: the final exam uses different data (`test`), so weights that are only lucky on `val` will not hold up.

In [ ]:
def scoreboard(weights):
    """Compares a set of weights with the default weights and the baselines, on VAL."""
    mine = val_ndcg(weights)
    rows = [("chronological baseline", baseline_val["chronological"]),
            ("popularity baseline", baseline_val["popularity"]),
            ("default weights (1, 1, 1)", default_val),
            ("YOUR weights", mine)]
    longest = max(value for _, value in rows)
    for name, value in rows:
        print(f"{name:26s} {value:.4f}  {'█' * round(40 * value / longest)}")
    if mine > max(default_val, *baseline_val.values()):
        print("\nBetter than the default and both baselines. You can move on.")
    else:
        print("\nNot better than the default yet. Change one knob and run again.")


# ===================== TODO: change these three numbers =====================
weights = {
    "freshness":  1.0,    # how much do you care that an item is NEW?
    "affinity":   0.0,    # ... that it matches what THIS user likes?
    "popularity": 1.0,    # ... that MANY people like it?
}
# ============================================================================
scoreboard(weights)

In [ ]:
assert set(weights) == {"freshness", "affinity", "popularity"} and all(v >= 0 for v in weights.values()) \
    and any(v > 0 for v in weights.values()), "weights: three numbers, none negative, at least one above 0"
mine = val_ndcg(weights)
assert mine > default_val, f"not better than the default weights yet ({mine:.4f} vs {default_val:.4f}): change a knob and re-run the task cell"
assert mine > max(baseline_val.values()), "your feed must also beat both baselines on val"
print(f"Checkpoint passed: val NDCG@10 = {mine:.4f} (default weights: {default_val:.4f}).")

## Where does the feed fail?

An average score hides people the feed serves badly. The cell below finds the **3 users with the lowest NDCG@10** on `val` and shows, for each one: how many likes they had, their favourite genres, the genres in their feed, and the genres of the items they actually liked later.

Read the output, then write **one plain sentence per user** about *why* the feed missed — for example "this user has very few likes, so we know almost nothing about them" or "their later likes were in genres they never liked before". Put the three sentences in the `RESULTS` block.

This matters for InstaKill: every new user starts with **no history at all** (the "cold start" problem), so you will meet exactly these cases on day one.

In [ ]:
per_user = evaluate(make_feed(VAL, signals_val, weights), VAL)["per_user"]
for user, value in sorted(per_user.items(), key=lambda kv: (kv[1], kv[0]))[:3]:
    def top_genres(item_ids):
        return Counter(g for i in item_ids for g in ITEMS[i].genres).most_common(3)
    history_items = [like.item_id for like in VAL.history if like.user_id == user]
    feed_items = build_feed(user, signals_val, ITEMS, VAL.now, weights)
    print(f"user {user}: NDCG={value:.3f}, likes in history={len(history_items)}")
    print("   history genres:  ", top_genres(history_items))
    print("   feed genres:     ", top_genres(feed_items))
    print("   liked later:     ", top_genres(VAL.relevant[user]))

## Final exam: `test` (run once)

Do not change `weights` or any signal after this cell. It evaluates your weights, the default weights and both baselines on `test`, and requires your NDCG@10 to be **higher than all three**. If it is not, go back to the task, adjust the knobs using `val`, and run this cell again — but do not tune on what `test` shows you. A setting that was only lucky on `val` can fail here; that is exactly why `test` exists.

In [ ]:
signals_test = build_signals(TEST.history, ITEMS, TEST.now)
final = evaluate(make_feed(TEST, signals_test, weights), TEST)
test_baselines = {name: evaluate(make(TEST), TEST) for name, make in
                  [("chronological", chronological_feed), ("popularity", popularity_feed)]}
signals_test_default = evaluate(make_feed(TEST, signals_test, default_weights), TEST)
test_baselines["default weights"] = signals_test_default

print(f"{'system':14s} {'NDCG@10':>8s} {'HitRate@10':>11s}")
for name, result in [*test_baselines.items(), ("yours", final)]:
    print(f"{name:14s} {result['ndcg']:8.4f} {result['hit_rate']:11.3f}")

assert final["ndcg"] > max(r["ndcg"] for r in test_baselines.values()), \
    "on test your weights must beat the default weights and both baselines: go back to the task, tune on val (not test), and run again"
print("Your weights beat the default weights and both baselines on test.")

## RESULTS

RESULTS
baseline:
- val: chronological NDCG@10 = 0.0235; popularity NDCG@10 = 0.1012.
- test: chronological NDCG@10 = 0.0110; popularity NDCG@10 = 0.0523.
solution:
- weights: freshness=1.0, affinity=0.0, popularity=1.0.
- val: solution NDCG@10 = 0.1729; default NDCG@10 = 0.1721.
- test: solution NDCG@10 = 0.1976, HitRate@10 = 0.600; default NDCG@10 = 0.1963.
runtime:
- build_feed timings: 1k = 6.7 ms, 10k = 71.5 ms, 100k = 728.9 ms; top-k on 100k = 51.3 ms versus full sort = 190.0 ms.
- With C candidate items and top-k, the heap-based ranking is O(C log k); signal construction is O(H + I), where H is history and I is the catalog size.
memory / model size:
- There is no trained model. build_signals stores one liked-item set and genre counter per user, recent likes per user, and one popularity counter per item.
3 failure cases:
- Users with very few likes provide too little history for reliable genre affinity.
- A user's later likes may shift to genres that were rare in their earlier history.
- Popular items can dominate recommendations even when they are only weakly related to a user's interests.
what I would try next:
- Add a recency-weighted user profile and evaluate it on InstaKill to improve cold-start and changing-interest behavior.
